In [ ]:
import pandas as pd

# =====================================================
# 1. 기존 BERTopic 결과 불러오기
# =====================================================

INPUT_FILE = "chelsea_cafe_bertopic_result.xlsx"
OUTPUT_FILE = "chelsea_cafe_bertopic_macro_result.xlsx"

# 저장할 때 sheet_name을 다르게 했을 수 있으니 우선 시트 확인
xls = pd.ExcelFile(INPUT_FILE)
print(xls.sheet_names)

# 보통 topic_labeled_data 또는 cafe_topic_labeled_data에 원본+topic 결과가 있음
SHEET_NAME = "cafe_topic_labeled_data" if "cafe_topic_labeled_data" in xls.sheet_names else xls.sheet_names[0]

topic_df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

print(topic_df.shape)
print(topic_df.columns.tolist())

display(topic_df.head())

In [ ]:
# # =====================================================
# # 2. topic_name 없으면 생성
# # =====================================================

# if "topic_name" not in topic_df.columns:
#     if "Name" in topic_df.columns:
#         topic_df["topic_name"] = topic_df["Name"]
#     else:
#         topic_df["topic_name"] = topic_df["topic"].astype(str)

# print(topic_df[["topic", "topic_name"]].drop_duplicates().head(30))

In [ ]:
# =====================================================
# topic_info에서 Representation 붙이기
# =====================================================

topic_info = pd.read_excel(INPUT_FILE, sheet_name="topic_info")

rep_map = topic_info.set_index("Topic")["Representation"].to_dict()

topic_df["Representation"] = topic_df["topic"].map(rep_map)

display(
    topic_df[["topic", "topic_name", "Representation"]]
    .drop_duplicates()
    .sort_values("topic")
    .head(30)
)

In [ ]:
# =====================================================
# 세부 토픽 -> 상위 토픽 매핑: 키워드 점수 기반
# =====================================================

macro_keywords = {
    "이적시장/선수보강": [
        "이적", "이적시장", "시장", "영입", "임대", "계약", "오피셜",
        "주급", "가격", "이적료", "바이아웃", "개인합의", "메디컬",
        "디렉터", "보드진", "방출", "판매", "매각", "재계약",
        "보강", "매물", "유망주", "콜업", "승격",

        # 포지션 보강
        "스트라이커", "공격수", "윙어", "공미", "수미", "미드필더",
        "센터백", "풀백", "골키퍼", "키퍼", "윙백",

        # 첼시 이적/영입 관련 핵심 선수
        "주앙페드루", "주앙 페드루", "주앙페드로", "주앙 페드로",
        "리암델랍", "리암 델랍", "델랍", "딜랩",
        "제이미기튼스", "제이미 기튼스", "기튼스", "기텐스",
        "에스테방윌리앙", "에스테방 윌리앙", "에스테방", "에스테반",
        "다리오에수구", "다리오 에수구", "에수구",
        "켄드리파에스", "켄드리 파에스", "파에스",
        "마마두사르", "마마두 사르", "사르",
        "요렐하토", "요렐 하토", "하토",
        "안드레이산투스", "안드레이 산투스", "산투스",
        "아론안셀미노", "아론 안셀미노", "안셀미노"
    ],

    "시즌 결과/순위/유럽대항전": [
        "순위", "승점", "잔류", "강등", "우승", "준우승",
        "챔스", "챔피언스리그", "유로파", "유로파리그",
        "컨퍼런스", "컨퍼런스리그", "대항전", "유럽대항전",
        "fa컵", "리그컵", "카라바오컵", "클럽월드컵",
        "결승", "4강", "8강", "16강", "탈락", "진출",
        "승리", "패배", "무승부", "연승", "연패", "무패",
        "마지막", "확정", "기록", "경쟁", "중위권",
        "상위권", "하위권", "강팀", "약팀", "빅6",
        "런던더비", "더비"
    ],

    "감독/전술 방향성": [
        "마레스카", "엔조마레스카", "엔조 마레스카",
        "마레스카볼", "마레스카 축구",
        "로시니어", "리암로시니어", "리암 로시니어",
        "포체티노", "램파드", "투헬", "콘테", "무리뉴",

        "전술", "압박", "전방압박", "빌드업", "후방빌드업",
        "역습", "점유율", "라인업", "선발", "교체", "로테이션",
        "전술변화", "포메이션", "쓰리백", "백스리", "포백",
        "윙백", "풀백", "중원", "수비라인", "하프스페이스",
        "간격", "연계", "패턴", "전개", "운영", "점유",
        "성향", "선임", "감독", "코치", "방향성", "전술가"
    ],

    "선수 기량/포지션 평가": [
        # 플레이/능력 관련 일반 키워드
        "플레이", "능력", "폼", "클래스", "왼발", "오른발",
        "속도", "스피드", "피지컬", "몸싸움", "민첩성",
        "헤더", "공격", "수비", "가담", "압박", "패스",
        "탈압박", "드리블", "크로스", "슈팅", "결정력",
        "골결", "득점력", "찬스", "침투", "연계",
        "포지션", "미드필더", "공미", "수미", "윙어",
        "스트라이커", "공격수", "풀백", "센터백",
        "키퍼", "골키퍼", "주전", "백업", "로테이션",
        "실수", "단점", "장점", "태도", "비판", "성장",

        # 골키퍼
        "로베르트산체스", "로베르트 산체스", "로버트산체스", "산체스",
        "필립요르겐센", "필립 요르겐센", "요르겐센", "조르겐센",
        "가브리엘슬로니나", "가브리엘 슬로니나", "슬로니나", "가가",
        "테디샤먼로우", "테디 샤먼로우", "샤먼로우",

        # 수비수
        "리스제임스", "리스 제임스", "리스", "제임스",
        "말로구스토", "말로 구스토", "구스토",
        "마르크쿠쿠렐라", "마르크 쿠쿠렐라", "쿠쿠렐라", "쿠쿠",
        "리바이콜윌", "리바이 콜윌", "콜윌",
        "웨슬리포파나", "웨슬리 포파나", "포파나",
        "브누아바디아실", "브누아 바디아실", "바디아실", "바디아쉴",
        "토신아다라비오요", "토신 아다라비오요", "토신", "아다라비오요",
        "트레보찰로바", "트레보 찰로바", "찰로바",
        "조쉬아체암퐁", "조쉬 아체암퐁", "아체암퐁", "아챔퐁",
        "마마두사르", "마마두 사르", "사르",
        "요렐하토", "요렐 하토", "하토",
        "아론안셀미노", "아론 안셀미노", "안셀미노",

        # 미드필더
        "모이세스카이세도", "모이세스 카이세도", "카이세도", "카이세두",
        "엔조페르난데스", "엔조 페르난데스", "엔소페르난데스", "엔소 페르난데스",
        "로메오라비아", "로메오 라비아", "라비아",
        "키어넌듀스버리홀", "키어넌 듀스버리홀", "듀스버리홀", "듀스버리 홀",
        "안드레이산투스", "안드레이 산투스", "산투스",
        "카니추쿠에메카", "카니 추쿠에메카", "추쿠에메카", "추쿠메카",
        "다리오에수구", "다리오 에수구", "에수구",
        "켄드리파에스", "켄드리 파에스", "파에스",

        # 공격수
        "콜팔머", "콜 팔머", "팔머",
        "페드로네투", "페드로 네투", "네투",
        "노니마두에케", "노니 마두에케", "마두에케", "마두에키",
        "미하일로무드릭", "미하일로 무드릭", "무드릭", "무드리크",
        "라힘스털링", "라힘 스털링", "스털링",
        "니콜라잭슨", "니콜라 잭슨", "잭슨",
        "크리스토퍼은쿤쿠", "크리스토퍼 은쿤쿠", "은쿤쿠", "은쿠쿠",
        "마르크기우", "마르크 기우", "기우", "귀우",
        "주앙페드루", "주앙 페드루", "주앙페드로", "주앙 페드로",
        "리암델랍", "리암 델랍", "델랍", "딜랩",
        "제이미기튼스", "제이미 기튼스", "기튼스", "기텐스",
        "에스테방윌리앙", "에스테방 윌리앙", "에스테방", "에스테반"
    ],

    "구단 운영/보드진 비판": [
        "보드진", "수뇌부", "구단주", "운영", "프런트", "프론트",
        "토드볼리", "토드 보엘리", "보엘리", "볼리",
        "에그발리", "클리어레이크", "블루코", "blueco",
        "무능", "실망", "최악", "위험", "충격",
        "돈", "투자", "재정", "ffp", "psr",
        "매각", "인수", "구단", "플랜", "프로젝트",
        "리빌딩", "개혁", "방향성", "스카우팅", "디렉터",
        "비전", "장기계약", "계약기간", "어린선수", "유망주정책"
    ],

    "부상/복귀 이슈": [
        "부상", "회복", "복귀", "재활", "의료진", "메디컬",
        "장기부상", "부상복귀", "관리", "결장", "아웃",
        "햄스트링", "근육", "무릎", "발목", "몸상태",
        "컨디션", "핏", "출전가능", "명단제외", "부상자",

        # 첼시 부상 이슈 자주 언급될 수 있는 선수
        "리스제임스", "리스 제임스", "리스", "제임스",
        "웨슬리포파나", "웨슬리 포파나", "포파나",
        "리바이콜윌", "리바이 콜윌", "콜윌",
        "로메오라비아", "로메오 라비아", "라비아",
        "크리스토퍼은쿤쿠", "크리스토퍼 은쿤쿠", "은쿤쿠",
        "미하일로무드릭", "미하일로 무드릭", "무드릭",
        "말로구스토", "말로 구스토", "구스토",
        "카니추쿠에메카", "카니 추쿠에메카", "추쿠에메카"
    ],

    "골키퍼/수비진 평가": [
        "키퍼", "골키퍼", "선방", "세이브", "실점",
        "클린시트", "빌드업", "킥", "킥력", "공중볼",
        "펀칭", "캐칭", "반응속도", "위치선정",
        "수비", "수비진", "센터백", "풀백", "윙백",
        "수비라인", "라인컨트롤", "커버", "대인마크",
        "태클", "블록", "걷어내기", "클리어링", "수비조직",

        # 골키퍼
        "로베르트산체스", "로베르트 산체스", "로버트산체스", "산체스",
        "필립요르겐센", "필립 요르겐센", "요르겐센", "조르겐센",
        "가브리엘슬로니나", "가브리엘 슬로니나", "슬로니나",
        "테디샤먼로우", "테디 샤먼로우", "샤먼로우",

        # 수비진
        "리스제임스", "리스 제임스", "리스", "제임스",
        "말로구스토", "말로 구스토", "구스토",
        "마르크쿠쿠렐라", "마르크 쿠쿠렐라", "쿠쿠렐라", "쿠쿠",
        "리바이콜윌", "리바이 콜윌", "콜윌",
        "웨슬리포파나", "웨슬리 포파나", "포파나",
        "브누아바디아실", "브누아 바디아실", "바디아실", "바디아쉴",
        "토신아다라비오요", "토신 아다라비오요", "토신", "아다라비오요",
        "트레보찰로바", "트레보 찰로바", "찰로바",
        "조쉬아체암퐁", "조쉬 아체암퐁", "아체암퐁", "아챔퐁",
        "마마두사르", "마마두 사르", "사르",
        "요렐하토", "요렐 하토", "하토",
        "아론안셀미노", "아론 안셀미노", "안셀미노"
    ],

    "공격진/득점력 평가": [
        "공격", "공격진", "득점", "득점력", "골결",
        "결정력", "찬스", "빅찬스", "슈팅", "마무리",
        "침투", "연계", "크로스", "드리블", "돌파",
        "스피드", "전환", "역습", "원톱", "스트라이커",
        "윙어", "공격수", "찬스메이킹", "박스", "박스안",

        "콜팔머", "콜 팔머", "팔머",
        "페드로네투", "페드로 네투", "네투",
        "노니마두에케", "노니 마두에케", "마두에케", "마두에키",
        "미하일로무드릭", "미하일로 무드릭", "무드릭", "무드리크",
        "라힘스털링", "라힘 스털링", "스털링",
        "니콜라잭슨", "니콜라 잭슨", "잭슨",
        "크리스토퍼은쿤쿠", "크리스토퍼 은쿤쿠", "은쿤쿠", "은쿠쿠",
        "마르크기우", "마르크 기우", "기우", "귀우",
        "주앙페드루", "주앙 페드루", "주앙페드로", "주앙 페드로",
        "리암델랍", "리암 델랍", "델랍", "딜랩",
        "제이미기튼스", "제이미 기튼스", "기튼스", "기텐스",
        "에스테방윌리앙", "에스테방 윌리앙", "에스테방", "에스테반"
    ],

    "중원/미드필더 평가": [
        "중원", "미드필더", "수미", "공미", "박스투박스",
        "탈압박", "패스", "전진패스", "키패스",
        "압박", "수비가담", "볼배급", "전개",
        "템포", "활동량", "피지컬", "세컨볼",
        "턴오버", "패스미스", "창의성", "볼소유",
        "빌드업", "전환패스", "압박회피",

        "모이세스카이세도", "모이세스 카이세도", "카이세도", "카이세두",
        "엔조페르난데스", "엔조 페르난데스", "엔소페르난데스", "엔소 페르난데스",
        "로메오라비아", "로메오 라비아", "라비아",
        "키어넌듀스버리홀", "키어넌 듀스버리홀", "듀스버리홀", "듀스버리 홀",
        "안드레이산투스", "안드레이 산투스", "산투스",
        "카니추쿠에메카", "카니 추쿠에메카", "추쿠에메카", "추쿠메카",
        "다리오에수구", "다리오 에수구", "에수구",
        "켄드리파에스", "켄드리 파에스", "파에스",
        "콜팔머", "콜 팔머", "팔머"
    ],

    "팬 반응/소통": [
        "감사", "덕분", "의견", "설명", "공감", "동의",
        "동감", "행복", "나이스", "수고", "좋아요",
        "응원", "팬", "팬들", "기대", "희망",
        "걱정", "불안", "화남", "빡침", "답답",
        "레전드", "근본", "낭만", "믿는다",
        "제발", "가자", "화이팅", "파이팅",
        "첼시", "블루스", "스탬포드브릿지", "스탬포드 브릿지"
    ]
}

def assign_macro_topic_score(topic_text):
    t = str(topic_text).lower()

    if "기타" in t or "-1" in t:
        return "기타/분류불가"

    scores = {}

    for macro, keywords in macro_keywords.items():
        matched = [k for k in keywords if k in t]
        scores[macro] = len(matched)

    # 가장 많이 매칭된 카테고리 선택
    best_macro = max(scores, key=scores.get)
    best_score = scores[best_macro]

    # 아무 키워드도 안 걸리면 기타
    if best_score == 0:
        return "기타/분류불가"

    return best_macro

In [ ]:
topic_df["macro_topic"] = topic_df["topic_name"].apply(assign_macro_topic_score)

display(
    topic_df[["topic", "topic_name", "macro_topic"]]
    .drop_duplicates()
    .sort_values("topic")
)

In [ ]:
def get_macro_scores(topic_text):
    t = str(topic_text).lower()

    scores = {}

    for macro, keywords in macro_keywords.items():
        matched = [k for k in keywords if k in t]
        scores[macro] = {
            "score": len(matched),
            "matched_keywords": matched
        }

    return scores


# topic별 점수 확인용 테이블 만들기
score_rows = []

for _, row in (
    topic_df[["topic", "Representation"]]
    .drop_duplicates()
    .sort_values("topic")
    .iterrows()
):
    if row["topic"] == -1:
        macro_topic = "기타/분류불가"
        scores = {}
    else:
        scores = get_macro_scores(row["Representation"])
        macro_topic = assign_macro_topic_score(row["Representation"])

    score_rows.append({
        "topic": row["topic"],
        "Representation": row["Representation"],
        "macro_topic": assign_macro_topic_score(row["Representation"]),
        "scores": scores
    })

score_df = pd.DataFrame(score_rows)

display(score_df)

In [ ]:
# # =====================================================
# # 5. 필요하면 topic 번호 기준 수동 보정
# # =====================================================

# manual_override = {
#     # 예시: topic 번호: "상위 토픽명"
#     # 7: "선수 기량/포지션 평가",
#     # 8: "이적시장/선수보강",
#     # 10: "이적시장/선수보강",
#     # 11: "이적시장/선수보강",
#     # 18: "이적시장/선수보강",
#     # 22: "이적시장/선수보강",
# }

# for topic_id, macro in manual_override.items():
#     topic_df.loc[topic_df["topic"] == topic_id, "macro_topic"] = macro

# display(
#     topic_df[["topic", "topic_name", "macro_topic"]]
#     .drop_duplicates()
#     .sort_values("topic")
# )

In [ ]:
# =====================================================
# 6. 38R 전후 상위 토픽 비교
# =====================================================

period_count = pd.crosstab(
    topic_df["topic_period"],
    topic_df["macro_topic"]
)

period_ratio = pd.crosstab(
    topic_df["topic_period"],
    topic_df["macro_topic"],
    normalize="index"
) * 100

display(period_count)
display(period_ratio.round(2))

In [ ]:
# =====================================================
# 7. 긍정/부정별 상위 토픽 비교
# =====================================================

pn_df = topic_df[topic_df["gpt_sentiment"].isin(["positive", "negative"])].copy()

sentiment_count = pd.crosstab(
    pn_df["gpt_sentiment"],
    pn_df["macro_topic"]
)

sentiment_ratio = pd.crosstab(
    pn_df["gpt_sentiment"],
    pn_df["macro_topic"],
    normalize="index"
) * 100

display(sentiment_count)
display(sentiment_ratio.round(2))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# =====================================================
# 한글 폰트 설정
# =====================================================

def set_korean_font():
    font_candidates = [
        "Malgun Gothic",      # Windows
        "AppleGothic",        # Mac
        "NanumGothic",        # Linux / Colab
        "Noto Sans CJK KR"
    ]

    installed_fonts = {f.name: f.fname for f in fm.fontManager.ttflist}

    for font in font_candidates:
        if font in installed_fonts:
            plt.rcParams["font.family"] = font
            plt.rcParams["axes.unicode_minus"] = False
            print(f"✅ 한글 폰트 설정 완료: {font}")
            return installed_fonts[font]

    print("⚠️ 한글 폰트를 찾지 못했습니다. Windows면 아래 직접 지정 코드를 사용하세요.")
    return None


font_path = set_korean_font()

In [ ]:
import re
from kiwipiepy import Kiwi

kiwi = Kiwi()

custom_words = [
    # 감독
    "마레스카", "로시니어",

    # 골키퍼
    "산체스", "요르겐센", "슬로니나",

    # 수비수
    "리스 제임스", "구스토", "쿠쿠렐라", "콜윌",
    "포파나", "바디아실", "토신", "찰로바",
    "아체암퐁", "사르", "하토", "안셀미노",

    # 미드필더
    "카이세도", "엔조 페르난데스", "라비아",
    "듀스버리홀", "안드레이 산투스",
    "추쿠에메카", "에수구", "파에스",

    # 공격수
    "팔머", "네투", "마두에케", "무드릭",
    "스털링", "잭슨", "은쿤쿠", "기우",
    "주앙 페드루", "델랍", "기튼스", "에스테방",

    # 팀명/조롱
    "첼시", "블루스", "첼시조롱", "첼애우", "첼붕이"
]

for word in custom_words:
    kiwi.add_user_word(word, "NNP")


stopwords = set([
    # [1] 기존 & 유튜브/커뮤니티 은어
    '진짜', '너무', '영상', '오늘', '경기', '그냥', '진심', 'ㅋㅋ', 'ㅎㅎ', 'ㅠㅠ', 'ㅜㅜ', 
    '이거', '저거', '그거', '뭔가', '정말', '지금', '다시', '계속', '가장', '제일', '조금', '정도','개추',
    
    # [2] 축구 일반 용어 (분석 인사이트가 없는 뻔한 단어)
    '축구', '하이라이트', '감독', '선수', '시즌', '리그', '전반', '후반', '골', '볼', '팬', '응원', '생각', '사람', '우리', '이번',
    
    # [3] 토트넘 지칭
    '첼시', '첼애우', '첼강딱',
    
    # [4] 타 팀 및 관련 선수 (노이즈 제거)
    '토트넘', '스퍼스', '닭집', '맨시티', '맨체스터시티', 
    '아스날', '아스널', '개스널', '리버풀', '전범풀', '맨유', '맹구', '맨체스터 유나이티드', 
    '뉴캐슬', '빌라', '에버튼', '울버햄튼', '황희찬', 
    '레알', '바르샤', '뮌헨', '김민재', '이강인', '파리', 'psg', '리즈',
    '웨스트햄', '아스톤빌라','웨햄','클럽',
    
    
    # [6] 형태소 분석(Okt) 후 흔히 남는 무의미한 두 글자 명사들
    '때문', '위해', '대한', '관련', '진짜' ,"작성","짧은","글", "질문", "삭제", "필수", "검색",
    "경기시작", "익일", "오전" ,"일반","일반글", "작성", "금지","게시","필독","성의","숙지","상단","등업",
    "카페", "댓글", "게시글",
    "팬", "회원", "작성", "게시", "필수", "일반", "삭제", "익일", "필독",
    "공지", "규정", "등업", "글쓰기", "제목"
])

stopwords = set(str(w).strip().lower() for w in stopwords)

def kiwi_tokenizer(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []

    for token in kiwi.tokenize(text):
        if token.tag in ["NNG", "NNP", "SL"]:
            word = token.form.strip().lower()

            if len(word) >= 2 and word not in stopwords:
                tokens.append(word)

    return tokens

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

# 색상
TOTAL_COLOR = "#D8E3F0"   # 전체 데이터: 연한 색
TOPIC_COLOR = "#132257"   # 특정 토픽: 진한 색

TEXT_COL = "topic_text" if "topic_text" in topic_df.columns else "text_clean"


def count_keywords(data):
    counter = Counter()

    for text in data[TEXT_COL].dropna().astype(str):
        counter.update(kiwi_tokenizer(text))

    return counter


# 전체 데이터 키워드 카운트
total_counter = count_keywords(topic_df)


def plot_topic_keywords(macro_topic, top_n=15):
    topic_data = topic_df[topic_df["macro_topic"] == macro_topic].copy()
    topic_counter = count_keywords(topic_data)

    top_words = [w for w, c in topic_counter.most_common(top_n)]

    plot_df = pd.DataFrame({
        "keyword": top_words,
        "topic_count": [topic_counter[w] for w in top_words],
        "total_count": [total_counter[w] for w in top_words]
    })

    x = range(len(plot_df))

    plt.figure(figsize=(12, 5))

    plt.bar(
        x,
        plot_df["total_count"],
        color=TOTAL_COLOR,
        label="전체 데이터 키워드 수"
    )

    plt.bar(
        x,
        plot_df["topic_count"],
        color=TOPIC_COLOR,
        label=f"{macro_topic} 키워드 수"
    )

    plt.xticks(x, plot_df["keyword"], rotation=45, ha="right")
    plt.title(f"{macro_topic} 주요 키워드 비교", fontsize=15, fontweight="bold")
    plt.ylabel("키워드 등장 횟수")
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()


# 가장 많은 상위 토픽 3개
top3_macros = topic_df["macro_topic"].value_counts().head(3).index.tolist()

for macro in top3_macros:
    plot_topic_keywords(macro, top_n=15)

In [ ]:
# 보고 싶은 상위 토픽명 직접 지정
TARGET_MACRO_TOPIC = "이적시장/선수보강"

plot_topic_keywords(
    TARGET_MACRO_TOPIC,
    top_n=15
)

In [ ]:
# =====================================================
# 8. 저장
# =====================================================

topic_macro_check = (
    topic_df
    .groupby(["topic", "topic_name", "macro_topic"])
    .size()
    .reset_index(name="count")
    .sort_values("topic")
)

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    topic_df.to_excel(writer, sheet_name="topic_macro_data", index=False)
    topic_macro_check.to_excel(writer, sheet_name="topic_macro_check", index=False)
    period_count.to_excel(writer, sheet_name="period_macro_count")
    period_ratio.round(2).to_excel(writer, sheet_name="period_macro_ratio")
    sentiment_count.to_excel(writer, sheet_name="sentiment_macro_count")
    sentiment_ratio.round(2).to_excel(writer, sheet_name="sentiment_macro_ratio")

print("저장 완료:", OUTPUT_FILE)